# 영화 흥행 예측 - Modeling

**팀: 9조 | 담당: 모델링 파트 | Data Science Term Project 2026**

## 목차
1. 환경 설정 및 데이터 로드
2. Classification — `Hit` 예측 (8개 모델)
3. Regression — `Rating` 예측 (7개 모델)
4. K-fold Cross Validation
5. Best-5 자동 탐색 함수 (오픈소스 기여)
6. 최종 결과 시각화 및 분석

## 모델링 전략
- **Classification 타겟**: `Hit` (이진, 클래스 불균형 10.3% : 89.7%) → 평가 지표는 F1, ROC-AUC 중심
- **Regression 타겟**: `Rating` (연속, 4.6~9.3)
- **Leakage 방지**: Hit이 Rating+Votes로 정의되므로, Regression의 X에서는 Hit을 제외하고 Classification의 X에서는 Rating을 제외함
- **CV**: StratifiedKFold (Classification), KFold (Regression), k=5

## 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_val_score, cross_validate, GridSearchCV
)

# Classification models (수업에서 학습한 모델만 사용)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score
)

# 시각화 설정
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('✅ Libraries loaded')

In [ ]:
# 전처리 완료된 데이터 로드
df = pd.read_csv('encoded_movie_dataset.csv')

# bool 컬럼 → int 변환 (모델 호환성)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(f'Shape: {df.shape}')
print(f'Hit 비율: {df["Hit"].mean()*100:.2f}% (Hit={df["Hit"].sum()}, Non-Hit={(df["Hit"]==0).sum()})')
df.head()

In [ ]:
# ============================================================
# Feature/Target 분리 — Leakage 방지
# ============================================================
# Identifier 컬럼 (분석 무관)
drop_cols = ['Movie Name', 'imdb_id']

# Classification용: Rating 제외 (Hit이 Rating에서 직접 파생됨)
X_clf = df.drop(columns=drop_cols + ['Hit', 'Rating'])
y_clf = df['Hit']

# Regression용: Hit 제외 (Hit이 Rating에서 파생됨)
X_reg = df.drop(columns=drop_cols + ['Hit', 'Rating'])
y_reg = df['Rating']

print('Classification:')
print(f'  X_clf shape: {X_clf.shape}, y_clf 분포: \n{y_clf.value_counts().to_dict()}')
print('\nRegression:')
print(f'  X_reg shape: {X_reg.shape}, y_reg 범위: {y_reg.min():.2f} ~ {y_reg.max():.2f}')
print(f'\n사용 피처 ({X_clf.shape[1]}개):')
print(X_clf.columns.tolist())

In [ ]:
# ============================================================
# Train/Test Split + Scaling
# ============================================================
# 수치형 컬럼 (스케일링 대상)
num_cols = ['Runtime', 'Metascore', 'Votes', 'Gross',
            'popularity', 'vote_average', 'release_year',
            'budget_adj', 'revenue_adj']

# Votes 로그 변환 (전처리 단계에서 빠진 경우 대비)
if X_clf['Votes'].max() > 100:  # 로그변환 전인지 체크
    X_clf = X_clf.copy()
    X_reg = X_reg.copy()
    X_clf['Votes'] = np.log1p(X_clf['Votes'])
    X_reg['Votes'] = np.log1p(X_reg['Votes'])
    print('✅ Votes log1p 변환 적용')

# Classification split (stratify로 불균형 유지)
X_clf_tr, X_clf_te, y_clf_tr, y_clf_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

# Regression split
X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# StandardScaler — fit은 train에만 (data leakage 방지)
scaler_clf = StandardScaler()
X_clf_tr[num_cols] = scaler_clf.fit_transform(X_clf_tr[num_cols])
X_clf_te[num_cols] = scaler_clf.transform(X_clf_te[num_cols])

scaler_reg = StandardScaler()
X_reg_tr[num_cols] = scaler_reg.fit_transform(X_reg_tr[num_cols])
X_reg_te[num_cols] = scaler_reg.transform(X_reg_te[num_cols])

print(f'\nClassification: train={X_clf_tr.shape}, test={X_clf_te.shape}')
print(f'Regression:     train={X_reg_tr.shape}, test={X_reg_te.shape}')
print(f'\nTrain Hit 비율: {y_clf_tr.mean()*100:.2f}%')
print(f'Test  Hit 비율: {y_clf_te.mean()*100:.2f}% (stratify로 유지됨)')

## 2. Classification — Hit 예측

6개 모델 비교: Logistic, KNN, Decision Tree, Random Forest, Gradient Boosting, XGBoost

**불균형 처리**: `class_weight='balanced'` 또는 `scale_pos_weight` 사용

**평가 지표**: Accuracy, Precision, Recall, F1, ROC-AUC (F1과 ROC-AUC가 주 지표)

In [ ]:
# 클래스 불균형 비율 계산 (XGBoost용)
neg_pos_ratio = (y_clf_tr == 0).sum() / (y_clf_tr == 1).sum()
print(f'Negative/Positive ratio: {neg_pos_ratio:.2f}')

clf_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'KNN': KNeighborsClassifier(n_neighbors=15),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=neg_pos_ratio,
        eval_metric='logloss', use_label_encoder=False,
        random_state=RANDOM_STATE, n_jobs=-1
    )
}

print(f'✅ {len(clf_models)}개 Classification 모델 준비')

In [ ]:
# ============================================================
# 모델별 학습 & 평가 + 5-Fold Stratified Cross Validation
# ============================================================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

clf_results = []
clf_trained = {}

for name, model in clf_models.items():
    print(f'⏳ Training: {name}...', end=' ')
    
    # 5-Fold CV (train 데이터로만)
    cv_results = cross_validate(
        model, X_clf_tr, y_clf_tr,
        cv=skf, scoring=scoring, n_jobs=-1
    )
    
    # 전체 train으로 fit → test로 최종 평가
    model.fit(X_clf_tr, y_clf_tr)
    y_pred = model.predict(X_clf_te)
    y_proba = model.predict_proba(X_clf_te)[:, 1]
    
    clf_trained[name] = model
    
    clf_results.append({
        'Model': name,
        'CV_Accuracy':  cv_results['test_accuracy'].mean(),
        'CV_Precision': cv_results['test_precision'].mean(),
        'CV_Recall':    cv_results['test_recall'].mean(),
        'CV_F1':        cv_results['test_f1'].mean(),
        'CV_ROC_AUC':   cv_results['test_roc_auc'].mean(),
        'CV_F1_std':    cv_results['test_f1'].std(),
        'Test_Accuracy':  accuracy_score(y_clf_te, y_pred),
        'Test_Precision': precision_score(y_clf_te, y_pred),
        'Test_Recall':    recall_score(y_clf_te, y_pred),
        'Test_F1':        f1_score(y_clf_te, y_pred),
        'Test_ROC_AUC':   roc_auc_score(y_clf_te, y_proba),
    })
    print('✅')

clf_df = pd.DataFrame(clf_results).sort_values('Test_F1', ascending=False).reset_index(drop=True)
print('\n' + '='*80)
print('Classification 결과 (Test F1 기준 정렬)')
print('='*80)
clf_df.round(4)

In [ ]:
# ============================================================
# Classification 결과 시각화
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# (1) 5-Fold CV vs Test 비교
ax = axes[0, 0]
x = np.arange(len(clf_df))
w = 0.35
ax.bar(x - w/2, clf_df['CV_F1'], w, label='5-Fold CV F1',
       yerr=clf_df['CV_F1_std'], capsize=4, color='steelblue', alpha=0.85)
ax.bar(x + w/2, clf_df['Test_F1'], w, label='Test F1', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(clf_df['Model'], rotation=35, ha='right')
ax.set_ylabel('F1 Score')
ax.set_title('Classification: 5-Fold CV F1 vs Test F1')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# (2) 지표별 히트맵
ax = axes[0, 1]
metric_cols = ['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_ROC_AUC']
sns.heatmap(
    clf_df.set_index('Model')[metric_cols],
    annot=True, fmt='.3f', cmap='YlGnBu', cbar_kws={'label': 'Score'}, ax=ax
)
ax.set_title('Classification Metrics Heatmap')

# (3) ROC Curves
ax = axes[1, 0]
for name, model in clf_trained.items():
    y_proba = model.predict_proba(X_clf_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_clf_te, y_proba)
    auc = roc_auc_score(y_clf_te, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=1.6)
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Test set)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)

# (4) Top 모델 Confusion Matrix
ax = axes[1, 1]
best_clf_name = clf_df.iloc[0]['Model']
best_clf = clf_trained[best_clf_name]
y_pred_best = best_clf.predict(X_clf_te)
cm = confusion_matrix(y_clf_te, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Hit', 'Hit'], yticklabels=['Non-Hit', 'Hit'], ax=ax)
ax.set_title(f'Confusion Matrix — Best Model: {best_clf_name}')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('classification_results.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n🏆 Best Classification Model: {best_clf_name}')
print(classification_report(y_clf_te, y_pred_best, target_names=['Non-Hit', 'Hit']))

In [ ]:
# Feature Importance — Tree 계열 모델 중 최고 성능 모델 사용
tree_models = ['XGBoost', 'Random Forest', 'Gradient Boosting', 'Decision Tree']
tree_best_name = next((m for m in clf_df['Model'] if m in tree_models), None)

if tree_best_name:
    tree_best = clf_trained[tree_best_name]
    importances = pd.DataFrame({
        'feature': X_clf_tr.columns,
        'importance': tree_best.feature_importances_
    }).sort_values('importance', ascending=False).head(15)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x='importance', y='feature', data=importances, palette='viridis')
    plt.title(f'Top 15 Feature Importances — {tree_best_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3. Regression — Rating 예측

5개 모델 비교: Linear, Decision Tree, Random Forest, Gradient Boosting, XGBoost

**평가 지표**: RMSE, MAE, R² (R²이 높을수록, RMSE/MAE가 낮을수록 좋음)

In [ ]:
reg_models = {
    'Linear Regression':   LinearRegression(),
    'Decision Tree':       DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestRegressor(
        n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting':   GradientBoostingRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    'XGBoost':             XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
}
print(f'✅ {len(reg_models)}개 Regression 모델 준비')

In [ ]:
# ============================================================
# 모델별 학습 & 평가 + 5-Fold Cross Validation
# ============================================================
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reg_scoring = ['neg_mean_squared_error', 'neg_mean_absolute_error', 'r2']

reg_results = []
reg_trained = {}

for name, model in reg_models.items():
    print(f'⏳ Training: {name}...', end=' ')
    
    cv_results = cross_validate(
        model, X_reg_tr, y_reg_tr,
        cv=kf, scoring=reg_scoring, n_jobs=-1
    )
    
    model.fit(X_reg_tr, y_reg_tr)
    y_pred = model.predict(X_reg_te)
    
    reg_trained[name] = model
    
    reg_results.append({
        'Model': name,
        'CV_RMSE':  np.sqrt(-cv_results['test_neg_mean_squared_error'].mean()),
        'CV_MAE':   -cv_results['test_neg_mean_absolute_error'].mean(),
        'CV_R2':    cv_results['test_r2'].mean(),
        'CV_R2_std': cv_results['test_r2'].std(),
        'Test_RMSE': np.sqrt(mean_squared_error(y_reg_te, y_pred)),
        'Test_MAE':  mean_absolute_error(y_reg_te, y_pred),
        'Test_R2':   r2_score(y_reg_te, y_pred),
    })
    print('✅')

reg_df = pd.DataFrame(reg_results).sort_values('Test_R2', ascending=False).reset_index(drop=True)
print('\n' + '='*80)
print('Regression 결과 (Test R² 기준 정렬)')
print('='*80)
reg_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) R² 비교
ax = axes[0]
x = np.arange(len(reg_df))
w = 0.35
ax.bar(x - w/2, reg_df['CV_R2'], w, label='5-Fold CV R²',
       yerr=reg_df['CV_R2_std'], capsize=4, color='steelblue', alpha=0.85)
ax.bar(x + w/2, reg_df['Test_R2'], w, label='Test R²', color='coral', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(reg_df['Model'], rotation=35, ha='right')
ax.set_ylabel('R² Score')
ax.set_title('Regression: R² Comparison')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# (2) RMSE/MAE 비교
ax = axes[1]
ax.bar(x - w/2, reg_df['Test_RMSE'], w, label='Test RMSE', color='salmon', alpha=0.85)
ax.bar(x + w/2, reg_df['Test_MAE'], w, label='Test MAE', color='teal', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(reg_df['Model'], rotation=35, ha='right')
ax.set_ylabel('Error')
ax.set_title('Regression: RMSE & MAE')
ax.legend(); ax.grid(axis='y', alpha=0.3)

# (3) Best 모델 Actual vs Predicted
ax = axes[2]
best_reg_name = reg_df.iloc[0]['Model']
best_reg = reg_trained[best_reg_name]
y_pred_best = best_reg.predict(X_reg_te)
ax.scatter(y_reg_te, y_pred_best, alpha=0.4, s=18, color='steelblue')
ax.plot([y_reg_te.min(), y_reg_te.max()],
        [y_reg_te.min(), y_reg_te.max()], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Rating'); ax.set_ylabel('Predicted Rating')
ax.set_title(f'Actual vs Predicted — {best_reg_name}')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('regression_results.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n🏆 Best Regression Model: {best_reg_name}')
print(f'   Test R²:   {reg_df.iloc[0]["Test_R2"]:.4f}')
print(f'   Test RMSE: {reg_df.iloc[0]["Test_RMSE"]:.4f}')
print(f'   Test MAE:  {reg_df.iloc[0]["Test_MAE"]:.4f}')

## 5. Best-5 자동 탐색 함수 (오픈소스 기여)

**목표**: 명세의 Open Source SW Contribution 요구사항을 단일 top-level 함수로 구현

단일 함수 `find_best_combinations()` 안에서 다음을 모두 수행한다:

1. **Preprocessing** — categorical **encoding** 여러 방법(onehot/label/ordinal) × data **scaling** 여러 방법(Standard/MinMax/Robust)
2. **Model training & testing** — 여러 모델 × 모델별 파라미터 조합
3. **Evaluation** — K-fold Cross Validation 기반 평가

→ 위 모든 조합 중 **Top-5 + 베스트 조합**을 반환

**4중 조합 축**: `encoder × scaler × model × hyperparameter`

Scikit-learn 스타일 docstring으로 작성하여 GitHub에 공개 (`best_combinations.py`).

In [ ]:
# ============================================================
# 오픈소스 함수: find_best_combinations()
# ============================================================
# 명세 요구사항(Open Source SW Contribution)을 모두 단일 top-level 함수에 구현:
#   (1) Preprocessing — categorical encoding 여러 방법 × data scaling 여러 방법
#   (2) Model training & testing — 여러 모델 × 모델별 파라미터 조합
#   (3) Evaluation — K-fold CV 기반 평가
#   → 위 조합들 중 Top-5 + 베스트 조합 반환
#
# 4중 조합 축: encoder × scaler × model × hyperparameter
#
# 함수 본체는 best_combinations.py 에 분리되어 있다 (오픈소스 배포용).
# 여기서는 import 하여 사용한다.

from best_combinations import find_best_combinations, apply_encoder

# (best_combinations.py 가 같은 폴더에 없다면, 아래 주석을 풀어 직접 정의해도 된다)
# %load best_combinations.py

print('✅ find_best_combinations() 로드 완료')

In [17]:
# ============================================================
# 실행: Classification Top-5 조합 찾기 (encoder × scaler × model × param)
# ============================================================
print('🔍 Classification Best-5 탐색 시작 (4중 조합)\n')

# 인코딩 전 원본 데이터를 사용 (함수가 내부에서 인코딩 수행)
df_raw = pd.read_csv('merged_movie_dataset.csv')
df_raw['Votes'] = np.log1p(df_raw['Votes'])   # Votes 로그 변환
npr = (df_raw['Hit'] == 0).sum() / (df_raw['Hit'] == 1).sum()

# 1축: Encoder (categorical encoding 방법)
clf_encoders = {'onehot': 'onehot', 'label': 'label', 'ordinal': 'ordinal'}

# 2축: Scaler (data scaling 방법)
clf_scalers = {
    'Standard': StandardScaler(),
    'MinMax':   MinMaxScaler(),
    'Robust':   RobustScaler(),
}

# 3축 + 4축: Model + 하이퍼파라미터 그리드
clf_search_space = {
    'LogisticRegression': (
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.1, 1.0, 10.0]}
    ),
    'RandomForest': (
        RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        {'n_estimators': [100, 200], 'max_depth': [10, 20]}
    ),
    'XGBoost': (
        XGBClassifier(
            eval_metric='logloss', use_label_encoder=False,
            scale_pos_weight=npr, random_state=RANDOM_STATE, n_jobs=-1
        ),
        {'n_estimators': [100, 200], 'max_depth': [4, 8]}
    ),
}

num_cols_raw = ['Runtime', 'Metascore', 'Votes', 'Gross', 'popularity',
                'vote_average', 'release_year', 'budget_adj', 'revenue_adj']

top5_clf = find_best_combinations(
    df_raw, target='Hit',
    categorical_cols=['Genre'],
    numeric_cols=num_cols_raw,
    encoders=clf_encoders,
    scalers=clf_scalers,
    models_with_params=clf_search_space,
    task='classification',
    cv=5, scoring='f1', top_k=5,
    drop_cols=['Movie Name', 'imdb_id', 'Rating'],   # Rating 제외 (leakage 방지)
    verbose=True,
)

print('\n' + '='*90)
print('🏆 Classification Top-5 (5-Fold CV F1 기준)')
print('='*90)
top5_clf.round(4)

Total combinations: 63 (encoders=3 x scalers=3 x param_points=7)

🏆 Classification Top-5 (5-Fold CV F1 기준)
   rank  encoder    scaler    model                                 params  mean_score  std_score
0     1    label  Standard  XGBoost  {'n_estimators': 100, 'max_depth': 8}      0.8542     0.0208
1     2  ordinal  Standard  XGBoost  {'n_estimators': 100, 'max_depth': 8}      0.8542     0.0208
2     3   onehot  Standard  XGBoost  {'n_estimators': 200, 'max_depth': 8}      0.8539     0.0164
3     4    label  Standard  XGBoost  {'n_estimators': 200, 'max_depth': 8}      0.8539     0.0188
4     5  ordinal  Standard  XGBoost  {'n_estimators': 200, 'max_depth': 8}      0.8539     0.0188


In [18]:
# ============================================================
# 실행: Regression Top-5 조합 찾기 (encoder × scaler × model × param)
# ============================================================
print('🔍 Regression Best-5 탐색 시작 (4중 조합)\n')

# 1축: Encoder
reg_encoders = {'onehot': 'onehot', 'label': 'label', 'ordinal': 'ordinal'}

# 2축: Scaler
reg_scalers = {
    'Standard': StandardScaler(),
    'MinMax':   MinMaxScaler(),
    'Robust':   RobustScaler(),
}

# 3축 + 4축: Model + 파라미터 (수업에서 학습한 모델만 사용)
reg_search_space = {
    'LinearRegression': (
        LinearRegression(),
        {}
    ),
    'DecisionTree': (
        DecisionTreeRegressor(random_state=RANDOM_STATE),
        {'max_depth': [5, 10, 15]}
    ),
    'GradientBoosting': (
        GradientBoostingRegressor(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1]}
    ),
}

# Regression은 Hit 제외 (Rating 예측이므로 Hit 제외, 타겟은 Rating)
top5_reg = find_best_combinations(
    df_raw, target='Rating',
    categorical_cols=['Genre'],
    numeric_cols=num_cols_raw,
    encoders=reg_encoders,
    scalers=reg_scalers,
    models_with_params=reg_search_space,
    task='regression',
    cv=5, scoring='r2', top_k=5,
    drop_cols=['Movie Name', 'imdb_id', 'Hit'],   # Hit 제외 (leakage 방지)
    verbose=True,
)

print('\n' + '='*90)
print('🏆 Regression Top-5 (5-Fold CV R² 기준)')
print('='*90)
top5_reg.round(4)

Total combinations: 54 (encoders=3 x scalers=3 x param_points=6)

🏆 Regression Top-5 (5-Fold CV R² 기준)
   rank  encoder    scaler             model                                       params  mean_score  std_score
0     1   onehot    MinMax  GradientBoosting  {'n_estimators': 200, 'learning_rate': 0.1}      0.8511     0.0074
1     2    label  Standard  GradientBoosting  {'n_estimators': 200, 'learning_rate': 0.1}      0.8510     0.0073
2     3  ordinal  Standard  GradientBoosting  {'n_estimators': 200, 'learning_rate': 0.1}      0.8510     0.0073
3     4  ordinal    Robust  GradientBoosting  {'n_estimators': 200, 'learning_rate': 0.1}      0.8509     0.0073
4     5    label    Robust  GradientBoosting  {'n_estimators': 200, 'learning_rate': 0.1}      0.8509     0.0073


## 6. 최종 결과 요약 및 저장

In [ ]:
# CSV로 모든 결과 저장 (보고서/PPT 작성용)
clf_df.to_csv('results_classification.csv', index=False)
reg_df.to_csv('results_regression.csv', index=False)
top5_clf.to_csv('top5_classification.csv', index=False)
top5_reg.to_csv('top5_regression.csv', index=False)

print('💾 저장 완료:')
print('  - results_classification.csv  (8개 모델 전체 결과)')
print('  - results_regression.csv      (7개 모델 전체 결과)')
print('  - top5_classification.csv     (스케일러×모델×파라미터 Top-5)')
print('  - top5_regression.csv         (스케일러×모델×파라미터 Top-5)')
print('  - classification_results.png  (시각화)')
print('  - regression_results.png      (시각화)')
print('  - feature_importance.png      (시각화)')

print('\n' + '='*70)
print('📊 최종 요약')
print('='*70)
print(f'\n[Classification] 🏆 Best: {clf_df.iloc[0]["Model"]}')
print(f'  Test F1:     {clf_df.iloc[0]["Test_F1"]:.4f}')
print(f'  Test ROC-AUC: {clf_df.iloc[0]["Test_ROC_AUC"]:.4f}')
print(f'  5-Fold CV F1: {clf_df.iloc[0]["CV_F1"]:.4f} ±{clf_df.iloc[0]["CV_F1_std"]:.4f}')

print(f'\n[Regression] 🏆 Best: {reg_df.iloc[0]["Model"]}')
print(f'  Test R²:   {reg_df.iloc[0]["Test_R2"]:.4f}')
print(f'  Test RMSE: {reg_df.iloc[0]["Test_RMSE"]:.4f}')
print(f'  Test MAE:  {reg_df.iloc[0]["Test_MAE"]:.4f}')

print(f'\n[오픈소스 함수] find_best_combinations()')
print(f'  - Classification: {top5_clf.iloc[0]["scaler"]} + {top5_clf.iloc[0]["model"]} + {top5_clf.iloc[0]["params"]}')
print(f'    → F1 = {top5_clf.iloc[0]["mean_score"]:.4f}')
print(f'  - Regression: {top5_reg.iloc[0]["scaler"]} + {top5_reg.iloc[0]["model"]} + {top5_reg.iloc[0]["params"]}')
print(f'    → R² = {top5_reg.iloc[0]["mean_score"]:.4f}')